<a href="https://colab.research.google.com/github/ale-balawi/bowari-site/blob/main/Kidney_Disease_Diagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

#fetch dataset
chronic_kidney_disease = fetch_ucirepo(id=336)

#data (as pandas dataframes)
X = chronic_kidney_disease.data.features
y = chronic_kidney_disease.data.targets

# metadata
print(chronic_kidney_disease.metadata)

# variable information
print(chronic_kidney_disease.variables)


{'uci_id': 336, 'name': 'Chronic Kidney Disease', 'repository_url': 'https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease', 'data_url': 'https://archive.ics.uci.edu/static/public/336/data.csv', 'abstract': 'This dataset can be used to predict the chronic kidney disease and it can be collected from the hospital nearly 2 months of period.', 'area': 'Other', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 400, 'num_features': 24, 'feature_types': ['Real'], 'demographics': ['Age'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2015, 'last_updated': 'Mon Mar 04 2024', 'dataset_doi': '10.24432/C5G020', 'creators': ['L. Rubini', 'P. Soundarapandian', 'P. Eswaran'], 'intro_paper': None, 'additional_info': {'summary': 'We use the following representation to collect the dataset\r\n                        age\t\t-\tage\t\r\n\t\t\tbp\t\t-\tblood pressure\r\n\t\t\tsg\t

In [ ]:
import pandas as pd
import numpy as np
from scipy.io import arff
import zipfile
import os

from sklearn.model_selection import train_test_split #separates training data from testing data
from sklearn.preprocessing import StandardScaler, LabelEncoder #normalizes numeric features
from sklearn.impute import SimpleImputer #converts categorical labels into numbers
from sklearn.preprocessing import LabelEncoder #fills in missing values

import torch
import torch.nn as nn


In [ ]:
device = torch.device("cpu")
if torch.cuda.is_available():
  device = torch.device("cuda")
#use cpu unless gpu (cuda)
elif torch.backends.mps.is_built() and torch.backends.mps.is_available():
  device = torch.device("mps")
#or we can check if your running it on a macbook with an apple silicon chip
print(device)

cuda


In [ ]:
print(X.shape)
print(y.shape)
X.head()
y.head()
#testing to see if data set returns (400,24)

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="ignore")
  #convert columns to rows

#seperate the data between numerical values and categorical
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))

#impute missing values using two stategies
num_imputer = SimpleImputer(strategy="median") #Some lab values can have outliers, median shifts from extreme values
cat_imputer = SimpleImputer(strategy="most_frequent") #If a feature is missing, assume the most common category

X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])
X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

#one-hot encode ts (convert categorical data into a numerical format). One-hot encoding coverts to values while avoiding fake ordering.
X = pd.get_dummies(X, columns=categorical_cols)

#encode target (like val = 1, notval = 0)
# Clean target manually (force true binary)
y = y.values.ravel()

# Remove whitespace just in case
y = np.array([str(label).strip() for label in y])

# Force binary mapping
y = np.where(y == 'ckd', 1, 0)

print("Unique labels after fix:", np.unique(y))


(400, 24)
(400, 1)
Numeric: 14
Categorical: 10
Unique labels after fix: [0 1]


/tmp/ipykernel_979/3295471546.py:8: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[col] = pd.to_numeric(X[col], errors="ignore")
/tmp/ipykernel_979/3295471546.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = pd.to_numeric(X[col], errors="ignore")
/tmp/ipykernel_979/3295471546.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[numeric_cols] = num_imputer.fit_transform(X[numeri

In [ ]:
#splitting the train and test data, then scaling it
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 #20% is for testing, other 80% for training
)

scaler = StandardScaler() #σ/(x−μ), bigger values aren't more important
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
print("Unique labels:", np.unique(y))
print("Min label:", y.min())
print("Max label:", y.max())

Unique labels: [0 1]
Min label: 0
Max label: 1


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test  = torch.tensor(X_test, dtype=torch.float32).to(device)

# Ensure labels are integers starting at 0
import numpy as np
y_train = torch.tensor(np.array(y_train), dtype=torch.long).to(device)
y_test  = torch.tensor(np.array(y_test), dtype=torch.long).to(device)

# Debug: check min/max labels
print("y_train min/max:", y_train.min().item(), y_train.max().item())
print("y_test min/max:", y_test.min().item(), y_test.max().item())

y_train -= y_train.min()
y_test  -= y_test.min()


Using device: cuda
y_train min/max: 0 1
y_test min/max: 0 1


In [ ]:
input_dim = X_train.shape[1]

class CKDNet(torch.nn.Module):
    def __init__(self):
        super().__init__()

        #64 down to 32 down to 2 (yes or nah)
        self.model = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),

            torch.nn.Linear(64, 32),
            torch.nn.ReLU(),

            torch.nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.model(x)

model = CKDNet().to(device)

In [ ]:
#Optimizer and Loss
criterion = torch.nn.CrossEntropyLoss() #crossentropyloss handles classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
#training loop
epochs = 100

for epoch in range(epochs):
    model.train()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if ((epoch + 1) % 10 == 0):
        print(f"Current Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Current Epoch [10/100], Loss: 0.5971
Current Epoch [20/100], Loss: 0.4553
Current Epoch [30/100], Loss: 0.3284
Current Epoch [40/100], Loss: 0.2158
Current Epoch [50/100], Loss: 0.1399
Current Epoch [60/100], Loss: 0.0838
Current Epoch [70/100], Loss: 0.0486
Current Epoch [80/100], Loss: 0.0366
Current Epoch [90/100], Loss: 0.0308
Current Epoch [100/100], Loss: 0.0209


In [ ]:
def CheckIfTsWorks(model, X_test, y_test, index):
    model.eval()

    with torch.no_grad():
        sample = X_test[index].unsqueeze(0)
        output = model(sample)
        predicted_class = torch.argmax(output, dim=1).item()
        true_class = y_test[index].item()

    print(f"Test Index: {index}")
    print(f"Predicted: {predicted_class}")
    print(f"Actual: {true_class}")

    print("\nAttributes (feature values):")
    for i, value in enumerate(X_test[index]):
        print(f"Feature {i}: {value.item()}")

In [ ]:
tempindex = 50
for i in range(3):
  CheckIfTsWorks(model, X_test, y_test, tempindex)
  tempindex += 1

Test Index: 50
Predicted: 1
Actual: 1

Attributes (feature values):
Feature 0: -0.17078281939029694
Feature 1: -0.4405781328678131
Feature 2: -0.50129634141922
Feature 3: -0.6812388896942139
Feature 4: -0.37752270698547363
Feature 5: -0.31173619627952576
Feature 6: -0.6277408003807068
Feature 7: -0.29526662826538086
Feature 8: 0.4532841145992279
Feature 9: -0.1366811841726303
Feature 10: -0.07740386575460434
Feature 11: -0.3020279109477997
Feature 12: -0.7436006665229797
Feature 13: -0.0609622485935688
Feature 14: -0.3447914123535156
Feature 15: 0.3447914123535156
Feature 16: -0.4754509925842285
Feature 17: 0.4754509925842285
Feature 18: 0.32751554250717163
Feature 19: -0.32751554250717163
Feature 20: 0.24413654208183289
Feature 21: -0.24413654208183289
Feature 22: 0.733799397945404
Feature 23: -0.733799397945404
Feature 24: -0.05598925054073334
Feature 25: -1.37218177318573
Feature 26: 1.3816986083984375
Feature 27: 0.3035624623298645
Feature 28: -0.3035624623298645
Feature 29: 0.5194

In [ ]:
def TestAccuracy(model, X_train, y_train):
  model.eval()  #switch to evaluation mode

  correct = 0
  total = y_test.size(0)

  with torch.no_grad():
        outputs = model(X_test) #pass entire test set at once
        predictions = torch.argmax(outputs, dim=1) #for each row, pick the answer with the highest predicted value

        correct = (predictions == y_test).sum().item() #counts how many values are true
  accuracy = (correct/total)
  print(f"Test Accuracy: {accuracy:.4f}")


In [ ]:
TestAccuracy(model, X_train, y_train)

Test Accuracy: 0.9875


In [ ]:
def PredictKidneyDisease(model):
    model.eval()

    print("\nEnter patient information:\n")

    # Collect Raw Inputs
    user_data = {
        "age": float(input("Age (years): ")),
        "bp": float(input("Blood Pressure (mm/Hg): ")),
        "sg": input("Specific Gravity (1.005,1.010,1.015,1.020,1.025): "),
        "al": input("Albumin (0-5): "),
        "su": input("Sugar (0-5): "),
        "rbc": input("Red Blood Cells (normal/abnormal): "),
        "pc": input("Pus Cell (normal/abnormal): "),
        "pcc": input("Pus Cell Clumps (present/notpresent): "),
        "ba": input("Bacteria (present/notpresent): "),
        "bgr": float(input("Blood Glucose Random (mgs/dl): ")),
        "bu": float(input("Blood Urea (mgs/dl): ")),
        "sc": float(input("Serum Creatinine (mgs/dl): ")),
        "sod": float(input("Sodium (mEq/L): ")),
        "pot": float(input("Potassium (mEq/L): ")),
        "hemo": float(input("Hemoglobin (gms): ")),
        "pcv": float(input("Packed Cell Volume: ")),
        "wbcc": float(input("White Blood Cell Count (cells/cmm): ")),
        "rbcc": float(input("Red Blood Cell Count (millions/cmm): ")),
        "htn": input("Hypertension (yes/no): "),
        "dm": input("Diabetes Mellitus (yes/no): "),
        "cad": input("Coronary Artery Disease (yes/no): "),
        "appet": input("Appetite (good/poor): "),
        "pe": input("Pedal Edema (yes/no): "),
        "ane": input("Anemia (yes/no): "),
    }

    # ----- Convert to DataFrame -----
    user_df = pd.DataFrame([user_data])

    # Ensure numeric columns are numeric
    for col in numeric_cols:
        user_df[col] = pd.to_numeric(user_df[col], errors="coerce")

    # ----- Apply SAME Imputers -----
    user_df[numeric_cols] = num_imputer.transform(user_df[numeric_cols])
    user_df[categorical_cols] = cat_imputer.transform(user_df[categorical_cols])

    # ----- One-hot encode -----
    user_df = pd.get_dummies(user_df)

    # ----- Match training column structure -----
    user_df = user_df.reindex(columns=X.columns, fill_value=0)

    # ----- Scale -----
    user_scaled = scaler.transform(user_df)

    # ----- Convert to tensor -----
    user_tensor = torch.tensor(user_scaled, dtype=torch.float32).to(device)

    # ----- Predict -----
    with torch.no_grad():
        output = model(user_tensor)
        probs = torch.softmax(output, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
        confidence = probs[0][prediction].item()

    if prediction == 1:
        print(f"\nPrediction: Chronic Kidney Disease Detected")
    else:
        print(f"\nPrediction: No Chronic Kidney Disease Detected")

    print(f"Model confidence: {confidence:.4f}")

In [ ]:
PredictKidneyDisease(model)


Enter patient information:

Age (years): 25
Blood Pressure (mm/Hg): 10
Specific Gravity (1.005,1.010,1.015,1.020,1.025): 1.015
Albumin (0-5): 0
Sugar (0-5): 0
Red Blood Cells (normal/abnormal): normal
Pus Cell (normal/abnormal): normal
Pus Cell Clumps (present/notpresent): notpresent
Bacteria (present/notpresent): notpresent
Blood Glucose Random (mgs/dl): 90
Blood Urea (mgs/dl): 14
Serum Creatinine (mgs/dl): 1
Sodium (mEq/L): 140
Potassium (mEq/L): 4.2
Hemoglobin (gms): 15
Packed Cell Volume: 45
White Blood Cell Count (cells/cmm): 7000
Red Blood Cell Count (millions/cmm): 5
Hypertension (yes/no): no
Diabetes Mellitus (yes/no): no
Coronary Artery Disease (yes/no): no
Appetite (good/poor): good
Pedal Edema (yes/no): no
Anemia (yes/no): no

Prediction: No Chronic Kidney Disease Detected
Model confidence: 0.9622
